# AiStock V8 数据获取测试 Notebook

本 Notebook 分模块测试 V8 系统全部数据源获取能力:

| 模块 | 数据源 | 端口/接口 | 测试内容 |
|------|--------|-----------|----------|
| 1 | TDX 标准端口 | 180.153.18.170:7709 | A股指数/股票K线、实时行情 |
| 2 | TDX 扩展端口 | 180.153.18.176:7721 | 期货K线、期权K线、合约列表、宏观指标 |
| 3 | AKShare | akshare API | 外盘期货实时/历史、辅助数据 |
| 4 | DatabaseReader | PostgreSQL | PE/PB估值、合约映射 |
| 5 | 期权PCR全流程 | TDX扩展+解析器 | 期权代码解析→合约发现→PCR计算 |

## 0. 环境初始化

In [1]:
import sys, os
from pathlib import Path

# 设置项目根目录 — 修改为你的实际路径
PROJECT_ROOT = Path().resolve().parent  # notebook 所在目录即项目根
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'项目根目录: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

# 检查核心依赖
deps = {}
for mod_name in ['pytdx', 'akshare', 'pandas', 'numpy', 'sqlalchemy', 'yaml']:
    try:
        mod = __import__(mod_name)
        ver = getattr(mod, '__version__', 'N/A')
        deps[mod_name] = f'OK ({ver})'
    except ImportError:
        deps[mod_name] = 'MISSING'

print('\n=== 依赖检查 ===')
for k, v in deps.items():
    status = '  ' if 'OK' in v else '**'
    print(f'  {status}{k}: {v}')

项目根目录: /home/ts/app/AiStock_v8
Python: 3.14.3 (main, Feb  6 2026, 23:41:13) [GCC 13.3.0]

=== 依赖检查 ===
    pytdx: OK (N/A)
    akshare: OK (1.18.62)
    pandas: OK (3.0.2)
    numpy: OK (2.3.5)
    sqlalchemy: OK (2.0.49)
    yaml: OK (6.0.3)


In [5]:
import pandas as pd
import numpy as np
import time
from datetime import datetime

# 通用测试辅助
def test_result(name, success, detail='', elapsed_ms=0):
    icon = 'PASS' if success else 'FAIL'
    timing = f' | {elapsed_ms:.0f}ms' if elapsed_ms else ''
    print(f'[{icon}] {name}{timing}  {detail}')
    return success

print('辅助函数就绪')

辅助函数就绪


---
## 1. TDX 标准端口测试 (7709)

测试 A股指数/股票 K线和实时行情获取

In [6]:
# ─── 1.1 直接 pytdx 连接测试 ───
from pytdx.hq import TdxHq_API

STD_HOST = '180.153.18.170'
STD_PORT = 7709

api = TdxHq_API()
t0 = time.time()
connected = api.connect(STD_HOST, STD_PORT)
elapsed = (time.time() - t0) * 1000

test_result('TDX标准端口连接', connected, f'{STD_HOST}:{STD_PORT}', elapsed)

if connected:
    # 市场证券数量
    count_sh = api.get_security_count(1)  # 上海
    count_sz = api.get_security_count(0)  # 深圳
    test_result('上海证券数量', count_sh > 0, f'count={count_sh}')
    test_result('深圳证券数量', count_sz > 0, f'count={count_sz}')

[PASS] TDX标准端口连接 | 230ms  180.153.18.170:7709
[PASS] 上海证券数量  count=27050
[PASS] 深圳证券数量  count=23321


In [7]:
# ─── 1.2 指数K线测试 ───
INDEX_TESTS = [
    ('000001', 1, '上证指数'),
    ('399001', 0, '深证成指'),
    ('399006', 0, '创业板指'),
    ('000300', 1, '沪深300'),
    ('000905', 1, '中证500'),
    ('000852', 1, '中证1000'),
]

print('=== 指数日线K线测试 ===')
index_results = {}
for code, market, name in INDEX_TESTS:
    try:
        t0 = time.time()
        df = api.to_df(api.get_index_bars(7, market, code, 0, 10))  # 日线, 10条
        elapsed = (time.time() - t0) * 1000
        ok = df is not None and not df.empty
        rows = len(df) if ok else 0
        test_result(f'{name}({code})', ok, f'{rows}条', elapsed)
        if ok:
            index_results[code] = df
            # 显示最近3条
            print(df[['open','close','high','low','vol']].tail(3).to_string())
            print()
    except Exception as e:
        test_result(f'{name}({code})', False, str(e)[:60])

=== 指数日线K线测试 ===
[PASS] 上证指数(000001) | 75ms  10条
      open    close     high      low           vol
7  4153.17  4153.88  4153.88  4153.17  5.005578e+06
8  4153.88  4153.88  4153.88  4153.88  5.877472e-39
9  4153.72  4152.57  4153.72  4152.57  1.197966e+08

[PASS] 深证成指(399001) | 77ms  10条
        open      close       high        low          vol
7  15856.690  15856.669  15856.690  15856.669    7357070.0
8  15856.669  15856.669  15856.669  15856.669     237240.0
9  15856.609  15856.609  15856.609  15856.609  155222016.0

[PASS] 创业板指(399006) | 86ms  10条
      open    close     high      low         vol
7  4020.06  4020.05  4020.06  4020.05   3576298.0
8  4020.05  4020.05  4020.05  4020.05    237895.0
9  4021.16  4021.16  4021.16  4021.16  79672112.0

[PASS] 沪深300(000300) | 76ms  10条
      open    close     high      low           vol
7  4921.96  4922.44  4922.44  4921.96  3.915528e+06
8  4922.44  4922.44  4922.44  4922.44  5.877472e-39
9  4922.29  4921.60  4922.29  4921.60  6.997840e+07

In [8]:
# ─── 1.3 实时行情测试 ───
print('=== 实时行情测试 ===')
QUOTE_TESTS = [
    (1, '600519'),  # 贵州茅台
    (1, '601318'),  # 中国平安
    (0, '000001'),  # 平安银行
    (0, '000858'),  # 五粮液
]

quotes = api.get_security_quotes(QUOTE_TESTS)
ok = quotes is not None and len(quotes) > 0
test_result('实时行情批量获取', ok, f'{len(quotes) if quotes else 0}条')

if ok:
    for q in quotes:
        print(f"  {q.get('code',''):6s} {q.get('name',''):8s} "
              f"价格={q.get('price',0):.2f} 涨跌={q.get('price',0)-q.get('last_close',0):+.2f}")

=== 实时行情测试 ===
[PASS] 实时行情批量获取  4条
  600519          价格=1285.88 涨跌=-4.32
  601318          价格=54.22 涨跌=+0.54
  000001          价格=10.68 涨跌=+0.00
  000858          价格=84.14 涨跌=+0.11


In [9]:
# 断开标准端口
api.disconnect()
print('标准端口已断开')

标准端口已断开


---
## 2. TDX 扩展端口测试 (7721)

测试期货K线、期权K线、合约列表发现、宏观指标

In [10]:
# ─── 2.1 扩展端口连接 ───
from pytdx.exhq import TdxExHq_API

EXT_HOST = '180.153.18.176'
EXT_PORT = 7721

ext_api = TdxExHq_API()
t0 = time.time()
ext_connected = ext_api.connect(EXT_HOST, EXT_PORT)
elapsed = (time.time() - t0) * 1000

test_result('TDX扩展端口连接', ext_connected, f'{EXT_HOST}:{EXT_PORT}', elapsed)

if ext_connected:
    total = ext_api.get_instrument_count()
    test_result('合约总数', total > 0, f'total={total}')

[PASS] TDX扩展端口连接 | 142ms  180.153.18.176:7721
[PASS] 合约总数  total=148899


In [14]:
# ─── 2.2 中金期货K线测试 ───
print('=== 中金期货K线测试 ===')
FUTURE_TESTS = [
    (47, 'IF300',  '沪深300主力'),
    (47, 'IC500',  '中证500主力'),
    (47, 'IM1000',  '中证1000主力'),
    (47, 'IH50',  '上证50主力'),
    (47, 'TFL8',  '5年期国债主力'),
    (30, 'AUL8',  '沪金主力'),
    (30, 'AGL8',  '沪银主力'),
    (30, 'CUL8',  '沪铜主力'),
]

for market, code, name in FUTURE_TESTS:
    try:
        t0 = time.time()
        df = ext_api.to_df(ext_api.get_instrument_bars(7, market, code, 0, 5))
        elapsed = (time.time() - t0) * 1000
        ok = df is not None and not df.empty
        rows = len(df) if ok else 0
        test_result(f'{name}({code})', ok, f'{rows}条', elapsed)
        if ok and 'close' in df.columns:
            print(f"  最新收盘: {df['close'].iloc[-1]}")
    except Exception as e:
        test_result(f'{name}({code})', False, str(e)[:60])

=== 中金期货K线测试 ===
[PASS] 沪深300主力(IF300) | 94ms  5条
  最新收盘: 4921.60009765625
[PASS] 中证500主力(IC500) | 91ms  5条
  最新收盘: 8703.890625
[PASS] 中证1000主力(IM1000) | 80ms  5条
  最新收盘: 8799.310546875
[PASS] 上证50主力(IH50) | 356ms  5条
  最新收盘: 2968.699951171875
[PASS] 5年期国债主力(TFL8) | 78ms  5条
  最新收盘: 106.46499633789062
[PASS] 沪金主力(AUL8) | 88ms  5条
  最新收盘: 998.7000122070312
[PASS] 沪银主力(AGL8) | 94ms  5条
  最新收盘: 18995.0
[PASS] 沪铜主力(CUL8) | 94ms  5条
  最新收盘: 105650.0


In [15]:
# ─── 2.3 合约列表发现 — 关键功能 ───
print('=== 合约列表发现测试 ===')

# 测试各市场的合约发现
MARKET_DISCOVERY = [
    (8,  '上海ETF期权',   '510050'),   # Market=8 上交所ETF期权
    (9,  '深圳ETF期权',   '159915'),   # Market=9 深交所ETF期权
    (48, '中金所期权',    'IO'),        # Market=48 中金期权
    (6,  '上期所商品期权', 'CU'),       # Market=6 上期所商品期权
    (47, '中金期货',      'IF'),        # Market=47 中金期货
]

discovery_results = {}
total_contracts = ext_api.get_instrument_count()

for market_code, label, prefix in MARKET_DISCOVERY:
    try:
        t0 = time.time()
        # 分批获取全量合约
        all_dfs = []
        batch_size = 2000
        for start in range(0, total_contracts + batch_size, batch_size):
            df = ext_api.get_instrument_info(start=start, count=batch_size)
            if df is not None and not df.empty:
                all_dfs.append(df)
        
        if all_dfs:
            combined = pd.concat(all_dfs, ignore_index=True)
            # 按市场筛选
            if 'market' in combined.columns:
                market_df = combined[combined['market'] == market_code]
            else:
                market_df = combined
            
            # 按标的前缀筛选
            if 'code' in market_df.columns:
                target_df = market_df[market_df['code'].str.startswith(prefix)]
            else:
                target_df = pd.DataFrame()
        else:
            target_df = pd.DataFrame()
        
        elapsed = (time.time() - t0) * 1000
        ok = not target_df.empty
        n = len(target_df)
        test_result(f'{label}({prefix}@{market_code})', ok, f'{n}个合约', elapsed)
        discovery_results[label] = target_df
        
        if ok:
            # 展示前5个合约代码
            codes = target_df['code'].head(5).tolist() if 'code' in target_df.columns else []
            print(f'  前5个: {codes}')
    except Exception as e:
        test_result(f'{label}({prefix}@{market_code})', False, str(e)[:80])

=== 合约列表发现测试 ===
[FAIL] 上海ETF期权(510050@8)  'list' object has no attribute 'empty'
[FAIL] 深圳ETF期权(159915@9)  'list' object has no attribute 'empty'
[FAIL] 中金所期权(IO@48)  'list' object has no attribute 'empty'
[FAIL] 上期所商品期权(CU@6)  'list' object has no attribute 'empty'
[FAIL] 中金期货(IF@47)  'list' object has no attribute 'empty'


In [19]:
# ─── 2.4 期权K线获取测试 ───
print('=== 期权K线获取测试 ===')

# 上海ETF期权: 510050 (上证50ETF) — Market=8
SH_OPTION_TESTS = [
    (8, '510050C6A02850', '50ETF购6A2850'),   # 假设的当月合约
    (8, '510050P6M02850', '50ETF沽6M2850'),
]

# 中金所期权: IO (沪深300) — Market=48
CFFEX_OPTION_TESTS = [
    (48, 'IO2606-C-4000', '沪深300购6月4000'),
    (7, 'IO8U0669', '沪深300沽6月4000'),
    (48, 'HO2606-C-2600', '上证50购6月2600'),
    (48, 'MO2606-C-6000', '中证1000购6月6000'),
]

# 深圳ETF期权: 159915 (创业板ETF) — Market=9
SZ_OPTION_TESTS = [
    (9, '159915C6A002000',  '创业板ETF购6A2000'),
    (9, '159915P6M002000A', '创业板ETF沽6M2000A[调整]'),
]

ALL_OPTION_TESTS = SH_OPTION_TESTS + CFFEX_OPTION_TESTS + SZ_OPTION_TESTS

for market, code, name in ALL_OPTION_TESTS:
    try:
        t0 = time.time()
        df = ext_api.to_df(ext_api.get_instrument_bars(7, market, code, 0, 3))
        elapsed = (time.time() - t0) * 1000
        ok = df is not None and not df.empty
        rows = len(df) if ok else 0
        test_result(f'{name}({code})', ok, f'{rows}条', elapsed)
        if ok:
            print(f"  最新: {df[['close','vol']].iloc[-1].to_dict()}")
    except Exception as e:
        test_result(f'{name}({code})', False, str(e)[:60])

=== 期权K线获取测试 ===
[FAIL] 50ETF购6A2850(510050C6A02850) | 67ms  0条
[FAIL] 50ETF沽6M2850(510050P6M02850) | 68ms  0条
[FAIL] 沪深300购6月4000(IO2606-C-4000) | 63ms  0条
[PASS] 沪深300沽6月4000(IO8U0669) | 2268ms  3条
[FAIL] 沪深300沽6月4000(IO8U0669)  "['vol'] not in index"
[FAIL] 上证50购6月2600(HO2606-C-2600) | 62ms  0条
[FAIL] 中证1000购6月6000(MO2606-C-6000) | 58ms  0条
[FAIL] 创业板ETF购6A2000(159915C6A002000) | 67ms  0条
[FAIL] 创业板ETF沽6M2000A[调整](159915P6M002000A) | 76ms  0条


In [20]:
# ─── 2.5 动态合约发现 + K线验证 ───
print('=== 动态合约发现 + K线验证 (510050) ===')

if '上海ETF期权' in discovery_results and not discovery_results['上海ETF期权'].empty:
    sh_opt_df = discovery_results['上海ETF期权']
    # 筛选 510050 标的
    etf_50 = sh_opt_df[sh_opt_df['code'].str.startswith('510050')]
    print(f'510050(上证50ETF)期权合约数: {len(etf_50)}')
    
    if not etf_50.empty:
        # 展示代码 + 名称
        display_cols = [c for c in ['code','name','market'] if c in etf_50.columns]
        print(etf_50[display_cols].head(20).to_string())
        
        # 随机选取3个合约获取K线
        sample_codes = etf_50['code'].sample(min(3, len(etf_50)), random_state=42).tolist()
        print(f'\n验证K线获取: {sample_codes}')
        for code in sample_codes:
            try:
                df = ext_api.to_df(ext_api.get_instrument_bars(7, 8, code, 0, 1))
                ok = df is not None and not df.empty
                test_result(f'K线({code})', ok, f'{len(df) if ok else 0}条')
            except Exception as e:
                test_result(f'K线({code})', False, str(e)[:50])
else:
    print('未获取到上海ETF期权合约列表, 跳过')

=== 动态合约发现 + K线验证 (510050) ===
未获取到上海ETF期权合约列表, 跳过


In [ ]:
# 断开扩展端口
ext_api.disconnect()
print('扩展端口已断开')

---
## 3. AKShare 外盘期货测试

测试 29 个外盘期货品种 + 辅助数据

In [21]:
import akshare as ak

print(f'akshare version: {ak.__version__}')

akshare version: 1.18.62


In [22]:
# ─── 3.1 外盘期货实时数据 (确认的5个核心品种) ───
print('=== ak.futures_foreign_commodity_realtime 测试 ===')

# 已确认可用的5品种
CONFIRMED_SYMBOLS = 'OIL,GC,CAD,NG,EUA'

try:
    t0 = time.time()
    df_rt = ak.futures_foreign_commodity_realtime(symbol=CONFIRMED_SYMBOLS)
    elapsed = (time.time() - t0) * 1000
    ok = df_rt is not None and not df_rt.empty
    test_result('futures_foreign_commodity_realtime', ok, f'{CONFIRMED_SYMBOLS} | {len(df_rt) if ok else 0}行', elapsed)
    if ok:
        print(df_rt.to_string())
except Exception as e:
    test_result('futures_foreign_commodity_realtime', False, str(e)[:80])

=== ak.futures_foreign_commodity_realtime 测试 ===
[PASS] futures_foreign_commodity_realtime | 645ms  OIL,GC,CAD,NG,EUA | 5行
         名称        最新价         人民币报价     涨跌额       涨跌幅       开盘价        最高价        最低价      昨日结算价  持仓量        买价        卖价      行情时间          日期
0     布伦特原油     96.400    654.218600  -3.810 -3.802016    97.500     97.720     94.110    100.210    0    96.280    96.310  16:31:08  2026-05-25
1   COMEX黄金   4594.663  31181.680449  38.263  0.839764  4566.000   4615.400   4565.100   4556.400    0  4591.700  4592.000  16:31:06  2026-05-25
2   LME铜3个月  13639.905   2976.035971 -27.595 -0.201902     0.000  13689.500  13551.000  13667.500    0     0.000     0.000  05:45:17  2026-05-25
3  NYMEX天然气      3.009    149.070223  -0.012 -0.397219     3.021      3.026      2.982      3.021    0     3.006     3.008  16:29:33  2026-05-25
4     欧洲碳排放     76.844    521.501806  -0.076 -0.098804    77.050     77.260     76.520     76.920    0    76.850    76.880  16:31:09  2026-05-25


In [30]:
# ─── 3.2 外盘期货历史数据 ───
print('=== ak.futures_foreign_hist 测试 ===')

HIST_TESTS = [
    ('WTI原油',  'CL'),
    ('COMEX黄金', 'GC'),
    ('天然气',   'NG'),
    ('COMEX铜',  'HG'),
    ('COMEX白银', 'SI'),
]

for ak_code, sym in HIST_TESTS:
    try:
        t0 = time.time()
        df = ak.futures_foreign_hist(symbol=sym)
        elapsed = (time.time() - t0) * 1000
        ok = df is not None and not df.empty
        rows = len(df) if ok else 0
        test_result(f'{ak_code}({sym})', ok, f'{rows}条', elapsed)
        if ok:
            # 最近3条
            print(f"  最新: {df.iloc[-1].to_dict()}")
    except Exception as e:
        test_result(f'{ak_code}({sym})', False, str(e)[:60])
    time.sleep(0.5)  # 限速

=== ak.futures_foreign_hist 测试 ===
[PASS] WTI原油(CL) | 948ms  7686条
  最新: {'date': Timestamp('2026-05-25 00:00:00'), 'open': 97.01, 'high': 93.9, 'low': 90.32, 'close': 92.39, 'volume': 0, 'position': 0, 's': 0.0, 'settlement': 0}
[PASS] COMEX黄金(GC) | 389ms  2589条
  最新: {'date': Timestamp('2026-05-25 00:00:00'), 'open': 4543.3, 'high': 4615.4, 'low': 4565.1, 'close': 4589.7, 'volume': 0, 'position': 0, 's': 0.0, 'settlement': 0}
[PASS] 天然气(NG) | 199ms  2579条
  最新: {'date': Timestamp('2026-05-25 00:00:00'), 'open': 3.033, 'high': 3.026, 'low': 2.982, 'close': 3.004, 'volume': 0, 'position': 0, 's': 0.0, 'settlement': 0}
[PASS] COMEX铜(HG) | 267ms  2590条
  最新: {'date': Timestamp('2026-05-25 00:00:00'), 'open': 638.0, 'high': 647.2, 'low': 638.35, 'close': 641.85, 'volume': 0, 'position': 0, 's': 0.0, 'settlement': 0}
[PASS] COMEX白银(SI) | 210ms  2589条
  最新: {'date': Timestamp('2026-05-25 00:00:00'), 'open': 75.88, 'high': 79.25, 'low': 76.705, 'close': 78.135, 'volume': 0, 'position': 0, 's

In [31]:
# ─── 3.3 辅助数据测试 ───
print('=== 辅助数据测试 ===')

AUX_TESTS = [
    ('bond_zh_us',      '中美利差',       lambda: ak.bond_zh_us_rate()),
    ('qvix_50etf',      '50ETF波动率指数', lambda: ak.option_current_em()),
]

for label, name, fetcher in AUX_TESTS:
    try:
        t0 = time.time()
        df = fetcher()
        elapsed = (time.time() - t0) * 1000
        ok = df is not None and not df.empty
        rows = len(df) if ok else 0
        test_result(f'{name}', ok, f'{rows}条', elapsed)
    except Exception as e:
        test_result(f'{name}', False, str(e)[:60])
    time.sleep(0.5)

=== 辅助数据测试 ===


  0%|          | 0/19 [00:00<?, ?it/s]

[PASS] 中美利差 | 4401ms  9262条


  0%|          | 0/268 [00:00<?, ?it/s]

[FAIL] 50ETF波动率指数  ('Connection aborted.', RemoteDisconnected('Remote end close


In [33]:
# ─── 3.4 V8 AKAdapter 模块测试 ───
print('=== V8 AKAdapter 模块测试 ===')

try:
    from data_service.ak_adapter import AKAdapter, CORE_SYMBOLS, ALL_SYMBOLS, FUTURES_SYMBOL_MAP
    
    ak_adapter = AKAdapter(
        rate_limit_interval=0.5,
        retry_count=2,
        retry_delay=1.0,
        cache_ttl=300.0,
    )
    test_result('AKAdapter初始化', True, f'{len(FUTURES_SYMBOL_MAP)}品种')
    
    # Core层批量获取
    print(f'\nCore品种({len(CORE_SYMBOLS)}): {CORE_SYMBOLS}')
    t0 = time.time()
    core_data = ak_adapter.get_futures_batch(CORE_SYMBOLS[:3])  # 先测3个
    elapsed = (time.time() - t0) * 1000
    ok_count = len(core_data)
    test_result('Core批量获取(3品种)', ok_count > 0, f'{ok_count}/3成功', elapsed)
    
    # 展示结果
    for sym, data in core_data.items():
        print(f"  {sym}: {data.get('name','')} 价格={data.get('price',0):.2f} 涨跌={data.get('pct_change',0):.2f}%")
    
    # 健康检查
    health = ak_adapter.health_check()
    test_result('AKAdapter健康检查', health.get('akshare', False), f"v{health.get('version','?')}")
    
except ImportError as e:
    test_result('AKAdapter导入', False, str(e))
except Exception as e:
    test_result('AKAdapter测试', False, str(e)[:80])

=== V8 AKAdapter 模块测试 ===
[FAIL] AKAdapter导入  No module named 'data_service'


---
## 4. DatabaseReader 测试

测试 PostgreSQL 估值数据 (PE/PB百分位) 和合约映射

In [39]:
# ─── 4.1 数据库连接测试 ───
print('=== DatabaseReader 连接测试 ===')

# 从 global_settings 获取数据库配置
try:
    from config.global_settings import DATABASE_ENGINES
    print(f'DATABASE_ENGINES keys: {list(DATABASE_ENGINES.keys())}')
    for k, v in DATABASE_ENGINES.items():
        if isinstance(v, str):
            print(f'  {k}: {v[:20]}...(string)')
        elif isinstance(v, dict):
            url = v.get('url', 'N/A')
            print(f'  {k}: {url[:30]}...(dict)')
except ImportError:
    DATABASE_ENGINES = {}
    print('global_settings 不可用, 使用默认配置')

try:
    from data_service.database_reader import DatabaseReader
    
    # 构建配置
    db_url = DATABASE_ENGINES.get('index_pe_db', '')
    if isinstance(db_url, str) and db_url:
        db_config = {'url': db_url}
    elif isinstance(db_url, dict):
        db_config = db_url
    else:
        db_config = None
    
    db_reader = DatabaseReader(
        db_config=db_config,
        engine_name='index_pe_db',
        fallback_on_error=True,
    )
    
    test_result('DatabaseReader初始化', True, f'connected={db_reader.is_connected}')
    
    # 健康检查
    health = db_reader.health_check()
    test_result('数据库健康检查', health.get('connected', False), str(health))
    
except ImportError as e:
    test_result('DatabaseReader导入', False, str(e))
except Exception as e:
    test_result('DatabaseReader测试', False, str(e)[:80])

=== DatabaseReader 连接测试 ===
global_settings 不可用, 使用默认配置
[FAIL] DatabaseReader导入  No module named 'data_service'


In [ ]:
# ─── 4.2 PE/PB估值数据测试 ───
print('=== PE/PB 估值数据测试 ===')

PE_TESTS = ['000300', '000905', '000852']

try:
    if db_reader.is_connected:
        for code in PE_TESTS:
            try:
                t0 = time.time()
                df = db_reader.get_index_pe(code, days=10)
                elapsed = (time.time() - t0) * 1000
                ok = not df.empty
                test_result(f'PE数据({code})', ok, f'{len(df)}条', elapsed)
                if ok:
                    print(f'  列: {list(df.columns)}')
                    print(f"  最新PE: {df.iloc[0].to_dict()}")
            except Exception as e:
                test_result(f'PE数据({code})', False, str(e)[:60])
    else:
        print('数据库未连接, 跳过PE测试')
        print('提示: 请检查 config/global_settings.py 中的 DATABASE_ENGINES 配置')
except NameError:
    print('db_reader 未创建, 跳过')

In [ ]:
# ─── 4.3 合约映射测试 ───
print('=== 合约映射测试 ===')

try:
    if db_reader.is_connected:
        for underlying in ['510050', 'IO']:
            try:
                df = db_reader.get_contract_mapping(underlying_code=underlying)
                ok = not df.empty
                test_result(f'合约映射({underlying})', ok, f'{len(df)}条')
                if ok:
                    print(f'  列: {list(df.columns)}')
                    print(f'  前3条: {df.head(3).to_string()}')
            except Exception as e:
                test_result(f'合约映射({underlying})', False, str(e)[:60])
    else:
        print('数据库未连接, 跳过合约映射测试')
except NameError:
    print('db_reader 未创建, 跳过')

---
## 5. 期权PCR数据获取全流程测试

测试 V8 核心能力: 期权代码解析 → 合约发现 → OI/Volume获取 → PCR计算

In [ ]:
# ─── 5.1 期权代码解析器测试 ───
print('=== OptionCodeParser 测试 ===')

from market_state_system.core.option_code_parser import OptionCodeParser

parser = OptionCodeParser()

# 三格式解析测试用例
PARSE_TESTS = [
    # (code, market_code, expected_underlying, expected_direction, expected_strike)
    ('510050C6A02850', 8, '510050', 'call', 2.850),     # 上交所ETF
    ('510050P6M02850', 8, '510050', 'put',  2.850),     # 上交所ETF Put
    ('510300C6B03800', 8, '510300', 'call', 3.800),     # 上交所300ETF
    ('159915C6A002000', 9, '159915', 'call', 2.000),    # 深交所ETF
    ('159915P6M002000A',9, '159915', 'put',  2.000),    # 深交所调整合约
    ('159919C6B004200', 9, '159919', 'call', 4.200),    # 深交所300ETF
    ('IO2606-C-4000',   7, 'IO',     'call', 4000.0),   # 中金所
    ('IO2606-P-4000',   7, 'IO',     'put',  4000.0),   # 中金所Put
    ('HO2606-C-2600',   7, 'HO',     'call', 2600.0),   # 上证50期权
    ('MO2606-C-6000',   7, 'MO',     'call', 6000.0),   # 中证1000期权
    ('CU2606-C-100000', 6, 'CU',     'call', 100000.0), # 商品期权
]

pass_count = 0
for code, mkt, exp_underlying, exp_dir, exp_strike in PARSE_TESTS:
    info = parser.parse(code, mkt)
    if info is None:
        test_result(f'解析({code})', False, '返回None')
        continue
    
    ok = (info.underlying == exp_underlying and 
          info.direction == exp_dir and 
          abs(info.strike_price - exp_strike) < 0.01)
    detail = f'{info.underlying} {info.direction} K={info.strike_price}'
    if not ok:
        detail += f' (期望: {exp_underlying} {exp_dir} K={exp_strike})'
    test_result(f'解析({code})', ok, detail)
    if ok:
        pass_count += 1

print(f'\n解析测试: {pass_count}/{len(PARSE_TESTS)} 通过')
print(f'解析器统计: {parser.stats}')

In [6]:
# ─── 5.2 合约代码构建(反向)测试 ───
print('=== 合约代码构建测试 ===')

BUILD_TESTS = [
    # (underlying, direction, year, month, strike, expected_code)
    ('510050', 'call', 2026, 1, 2.850, '510050C6A2850'),
    ('510050', 'put',  2026, 1, 2.850, '510050P6M2850'),
    ('159915', 'call', 2026, 2, 2.000, '159915C6B2000'),
]

for underlying, direction, year, month, strike, expected in BUILD_TESTS:
    code = OptionCodeParser.build_etf_code(underlying, direction, year, month, strike)
    ok = code == expected
    test_result(f'构建({underlying}{direction[0].upper()}{year}{month}K{strike})', ok,
                f'got={code} expected={expected}')

# CFFEX构建
cffex_code = OptionCodeParser.build_cffex_code('IO', 2026, 6, 'call', 4000)
test_result('CFFEX构建(IO)', cffex_code == 'IO2606-C-4000', f'got={cffex_code}')

=== 合约代码构建测试 ===


NameError: name 'OptionCodeParser' is not defined

In [ ]:
# ─── 5.3 TDXAdapter 模块化测试 ───
print('=== TDXAdapter 模块化测试 ===')

try:
    from data_service.tdx_adapter import TDXAdapter, MarketType, BarCategory
    
    tdx = TDXAdapter(
        std_host='180.153.18.170',
        std_port=7709,
        ext_host='180.153.18.176',
        ext_port=7721,
        pool_size=2,
    )
    test_result('TDXAdapter初始化', True)
    
    # 健康检查
    t0 = time.time()
    health = tdx.health_check()
    elapsed = (time.time() - t0) * 1000
    test_result('双端口健康检查', 
                health.get('standard_port', False) or health.get('extension_port', False),
                str(health), elapsed)
    
    # 指数K线
    t0 = time.time()
    df_idx = tdx.get_index_daily('000300', market_type=MarketType.INDEX_SH, count=5)
    elapsed = (time.time() - t0) * 1000
    test_result('TDXAdapter.指数K线(沪深300)', not df_idx.empty, f'{len(df_idx)}条', elapsed)
    
    # 期货K线
    t0 = time.time()
    df_fut = tdx.get_future_daily('IFM0', market_type=MarketType.FUTURE_ZJ, count=5)
    elapsed = (time.time() - t0) * 1000
    test_result('TDXAdapter.期货K线(IF主力)', not df_fut.empty, f'{len(df_fut)}条', elapsed)
    
    # 合约列表
    t0 = time.time()
    df_inst = tdx.get_instrument_info(market=48)  # 中金期权
    elapsed = (time.time() - t0) * 1000
    io_count = len(df_inst[df_inst['code'].str.startswith('IO')]) if not df_inst.empty and 'code' in df_inst.columns else 0
    test_result('TDXAdapter.合约列表(中金期权IO)', io_count > 0, f'{io_count}个IO合约', elapsed)
    
    tdx.close()
    
except Exception as e:
    test_result('TDXAdapter测试', False, str(e)[:100])

In [ ]:
# ─── 5.4 ETF PCR 端到端测试 (510050) ───
print('=== 50ETF PCR 端到端测试 ===')

try:
    # Step 1: 合约发现
    t0 = time.time()
    df_all = tdx_adapter_module.get_instrument_info(market=8)  # 上交所ETF期权
    elapsed = (time.time() - t0) * 1000
    
    # 重新连接 (如果上面已关闭)
    from data_service.tdx_adapter import TDXAdapter, MarketType, BarCategory
    tdx_pcr = TDXAdapter(std_host='180.153.18.170', std_port=7709,
                         ext_host='180.153.18.176', ext_port=7721, pool_size=2)
    
    # Step 1: 合约发现
    df_all = tdx_pcr.get_instrument_info(market=8)
    if df_all.empty:
        print('未获取到市场8合约, 跳过PCR测试')
    else:
        etf50_contracts = df_all[df_all['code'].str.startswith('510050')]
        print(f'Step1: 510050合约数 = {len(etf50_contracts)}')
        
        # Step 2: 解析合约
        contracts_info = [{'code_name': c, 'market_code': 8} 
                         for c in etf50_contracts['code'].tolist()]
        parsed = parser.parse_batch(contracts_info)
        target = [p for p in parsed if p.underlying == '510050']
        calls = [p for p in target if p.is_call]
        puts = [p for p in target if p.is_put]
        adjusted = [p for p in target if p.is_adjusted]
        print(f'Step2: 解析成功 {len(target)}/{len(contracts_info)} | Call={len(calls)} Put={len(puts)} 调整={len(adjusted)}')
        
        # Step 3: 采样获取OI/Volume (取前5个Call + 前5个Put)
        call_vols, put_vols = [], []
        for info in calls[:5]:
            df = tdx_pcr.get_bars('option_sh', info.code_name, BarCategory.DAILY, 0, 1)
            if not df.empty:
                vol = int(df.iloc[-1].get('volume', 0))
                call_vols.append(vol)
                print(f'  Call {info.code_name} K={info.strike_price:.3f} Vol={vol}')
        
        for info in puts[:5]:
            df = tdx_pcr.get_bars('option_sh', info.code_name, BarCategory.DAILY, 0, 1)
            if not df.empty:
                vol = int(df.iloc[-1].get('volume', 0))
                put_vols.append(vol)
                print(f'  Put  {info.code_name} K={info.strike_price:.3f} Vol={vol}')
        
        # Step 4: 计算PCR
        total_call_vol = sum(call_vols)
        total_put_vol = sum(put_vols)
        pcr_vol = total_put_vol / total_call_vol if total_call_vol > 0 else 0
        print(f'\nStep4: PCR(Volume采样) = {pcr_vol:.4f} (Call={total_call_vol}, Put={total_put_vol})')
        test_result('50ETC PCR采样计算', pcr_vol > 0, f'PCR={pcr_vol:.3f}')
    
    tdx_pcr.close()
    
except Exception as e:
    test_result('PCR端到端测试', False, str(e)[:100])

In [5]:
# ─── 5.5 CFFEX PCR 测试 (IO 沪深300) ───
print('=== IO PCR 端到端测试 ===')

try:
    from data_service.tdx_adapter import TDXAdapter, MarketType, BarCategory
    tdx_cffex = TDXAdapter(std_host='180.153.18.170', std_port=7709,
                           ext_host='180.153.18.176', ext_port=7721, pool_size=2)
    
    # Step 1: 中金所期权合约发现 (Market=48)
    df_cffex = tdx_cffex.get_instrument_info(market=48)
    if df_cffex.empty:
        print('未获取到中金所合约, 跳过')
    else:
        io_contracts = df_cffex[df_cffex['code'].str.startswith('IO')]
        print(f'Step1: IO合约数 = {len(io_contracts)}')
        if not io_contracts.empty:
            print(f'  示例代码: {io_contracts["code"].head(5).tolist()}')
        
        # Step 2: 解析
        contracts_info = [{'code_name': c, 'market_code': 7} 
                         for c in io_contracts['code'].tolist()]
        parsed = parser.parse_batch(contracts_info)
        io_parsed = [p for p in parsed if p.variety == 'IO']
        calls = [p for p in io_parsed if p.is_call]
        puts = [p for p in io_parsed if p.is_put]
        print(f'Step2: 解析成功 {len(io_parsed)}/{len(contracts_info)} | Call={len(calls)} Put={len(puts)}')
        
        # Step 3: 采样获取Volume
        call_vols, put_vols = [], []
        for info in calls[:3]:
            df = tdx_cffex.get_bars('option_zj', info.code_name, BarCategory.DAILY, 0, 1)
            if not df.empty:
                vol = int(df.iloc[-1].get('volume', 0))
                call_vols.append(vol)
                print(f'  Call {info.code_name} K={info.strike_price:.0f} Vol={vol}')
        
        for info in puts[:3]:
            df = tdx_cffex.get_bars('option_zj', info.code_name, BarCategory.DAILY, 0, 1)
            if not df.empty:
                vol = int(df.iloc[-1].get('volume', 0))
                put_vols.append(vol)
                print(f'  Put  {info.code_name} K={info.strike_price:.0f} Vol={vol}')
        
        total_call_vol = sum(call_vols)
        total_put_vol = sum(put_vols)
        pcr_vol = total_put_vol / total_call_vol if total_call_vol > 0 else 0
        print(f'\nPCR(Volume采样) = {pcr_vol:.4f}')
        test_result('IO PCR采样计算', pcr_vol > 0, f'PCR={pcr_vol:.3f}')
    
    tdx_cffex.close()
    
except Exception as e:
    test_result('CFFEX PCR测试', False, str(e)[:100])

=== IO PCR 端到端测试 ===


NameError: name 'test_result' is not defined

---
## 6. V8 完整模块集成测试

In [4]:
# ─── 6.1 DataLoaderService 集成测试 ───
print('=== DataLoaderService 集成测试 ===')

try:
    from data_service import TDXAdapter, AKAdapter, DatabaseReader, DataLoaderService
    
    # 初始化三大数据源
    tdx = TDXAdapter(std_host='180.153.18.170', std_port=7709,
                     ext_host='180.153.18.176', ext_port=7721, pool_size=2)
    ak_adp = AKAdapter(rate_limit_interval=0.5, retry_count=2)
    
    try:
        from config.global_settings import DATABASE_ENGINES
        db_url = DATABASE_ENGINES.get('index_pe_db', '')
        db_config = {'url': db_url} if isinstance(db_url, str) and db_url else None
    except ImportError:
        db_config = None
    
    db = DatabaseReader(db_config=db_config, engine_name='index_pe_db', fallback_on_error=True)
    
    # DataLoaderService
    loader = DataLoaderService(
        tdx_adapter=tdx,
        ak_adapter=ak_adp,
        db_reader=db,
    )
    test_result('DataLoaderService初始化', True)
    
    # 按段加载测试 (只测试关键段)
    print('\n--- 按段加载测试 ---')
    
    # 指数数据
    t0 = time.time()
    idx_data = loader.load_section('index_data')
    elapsed = (time.time() - t0) * 1000
    idx_ok = isinstance(idx_data, dict) and len(idx_data) > 0
    valid_idx = sum(1 for v in idx_data.values() if v is not None and not v.empty) if idx_ok else 0
    test_result('指数数据加载', idx_ok, f'{valid_idx}个有效指数', elapsed)
    
    # 期货数据
    t0 = time.time()
    fut_data = loader.load_section('futures_data')
    elapsed = (time.time() - t0) * 1000
    fut_ok = isinstance(fut_data, dict) and len(fut_data) > 0
    valid_fut = sum(1 for v in fut_data.values() if v is not None and not v.empty) if fut_ok else 0
    test_result('期货数据加载', fut_ok, f'{valid_fut}个有效品种', elapsed)
    
    # 海外期货
    t0 = time.time()
    ov_data = loader.load_section('overseas_futures')
    elapsed = (time.time() - t0) * 1000
    ov_ok = isinstance(ov_data, dict) and len(ov_data) > 0
    test_result('海外期货加载', ov_ok, f'{len(ov_data) if ov_ok else 0}个品种', elapsed)
    
    tdx.close()
    
except Exception as e:
    test_result('DataLoaderService集成', False, str(e)[:100])

=== DataLoaderService 集成测试 ===


NameError: name 'test_result' is not defined

In [3]:
# ─── 6.2 质量报告 ───
print('=== 数据质量报告 ===')

try:
    report = loader.get_quality_report()
    for section, info in report.items():
        print(f'\n[{section}]')
        if isinstance(info, dict):
            for k, v in info.items():
                print(f'  {k}: {v}')
    
    # 健康摘要
    health = loader.get_health_summary()
    print(f'\n=== 健康摘要 ===')
    print(f'数据源: {health.get("data_sources", {})}')
    print(f'质量: {health.get("quality", {})}')
    
except Exception as e:
    print(f'质量报告获取失败: {e}')

=== 数据质量报告 ===
质量报告获取失败: name 'loader' is not defined


---
## 7. 测试汇总

In [2]:
print('=' * 60)
print('AiStock V8 数据获取测试汇总')
print('=' * 60)
print(f'测试时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()
print('测试模块:')
print('  [1] TDX标准端口(7709) — 指数K线 + 实时行情')
print('  [2] TDX扩展端口(7721) — 期货/期权K线 + 合约发现')
print('  [3] AKShare — 外盘期货(29品种) + 辅助数据')
print('  [4] DatabaseReader — PE/PB估值 + 合约映射')
print('  [5] 期权PCR全流程 — 代码解析→合约发现→PCR计算')
print('  [6] DataLoaderService集成测试')
print()
print('关键验证点:')
print('  - TDX双端口连接: 标准(7709) + 扩展(7721)')
print('  - 三格式期权代码解析: 上交所(8) + 深交所(9) + 中金所(7)')
print('  - 动态合约发现: get_instrument_info() 替代静态代码表')
print('  - AKShare实时商品: OIL,GC,CAD,NG,EUA')
print('  - PCR采样计算: Volume/OI 分离')
print('=' * 60)

AiStock V8 数据获取测试汇总


NameError: name 'datetime' is not defined